In [1]:
from peft import PeftModel
from transformers import T5Tokenizer, T5EncoderModel
from clalign.alignment import ProteinSeq, AlignmentResult, align_core
from clalign.plm import PLM
from clalign.metrics import f1score, hec_acc, hec_sov

/home/yrh/CLAlign/.conda/envs/CLAlign-Release/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cd ..

/home/yrh/CLAlign


In [3]:
tokenizer = T5Tokenizer.from_pretrained('Rostlab/ProstT5')
model = T5EncoderModel.from_pretrained('Rostlab/ProstT5').cuda()

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195/195 [00:00<00:00, 15733.48it/s]


In [4]:
model = PeftModel.from_pretrained(model, 'src/clalign/CLAlign-ProstT5')
plm = PLM(tokenizer, model, 512)

In [5]:
def align(seq1, seq2):
    embs = plm.get_embs([seq1, seq2])
    return align_core(seq1, seq2, embs[0] @ embs[1].T)

In [6]:
import numpy as np
from pathlib import Path

def test(data):
    name_list, f_list, tms, hacc_list, hsov_list = [], [], [], [], []
    Path(f'results/{data}/clalign-prostt5').mkdir(parents=True, exist_ok=True)
    with open(f'data/{data}_hec.csv') as fp, open(f'results/{data}/clalign-prostt5.txt', 'w') as fout:
        fp.readline()
        for line in fp:
            name, f1, f2, s1, s2, hec1, hec2, aln1, aln2 = line.strip().split(',')
            name_list.append(name)
            manual = AlignmentResult(seq1 := ProteinSeq(s1), seq2 := ProteinSeq(s2), aln1, aln2)
            aln_res = align(seq1, seq2)
            f_list.append(f_ := f1score(manual, aln_res))         
            if len(seq1) == len(hec1) and len(seq2) == len(hec2):
                 hacc_list.append(hacc_ := hec_acc(aln_res, hec1, hec2))
                 hsov_list.append(hsov_ := hec_sov(aln_res, hec1, hec2))
            else:
                 print('hec not match')
                 hacc_ = hsov_ = 0.0
            with open(out_:=(f'results/{data}/clalign-prostt5/{name}.txt'), 'w') as faln:
                print('>p1', file=faln)
                print(aln_res.aln1, file=faln)
                print('>p2', file=faln)
                print(aln_res.aln2, file=faln)
            out = !TMalign data/{data}/{name}/{f1} data/{data}/{name}/{f2} -I {out_} -a T
            tms.append(s_:=float(out[17][10:17]))
            print(f'{name}: P: {f_[0]:.3f}, R: {f_[1]:.3f}, F: {f_[2]:.3f}, S: {s_:.5f}, HEC ACC: {hacc_:.5f}, HEC SOV: {hsov_:.5f}')
            print(name, f_[0], f_[1], f_[2], s_, hacc_, hsov_, file=fout, sep='\t')
        p_, r_, f_ = np.asarray(f_list).mean(axis=0)
        print(f'total: {len(f_list)}, P: {p_:.3f}, R: {r_:.3f}, F: {f_:.3f}, TM-score: {np.mean(tms):.5f}, HEC ACC: {np.mean(hacc_list):.5f}, HEC SOV: {np.mean(hsov_list):.5f}')
        return tms

In [7]:
malidup_tms = test('malidup')

d19hca_: P: 0.265, R: 0.307, F: 0.284, S: 0.27769, HEC ACC: 0.26255, HEC SOV: 0.61539
d1a4pa_: P: 0.750, R: 0.750, F: 0.750, S: 0.49735, HEC ACC: 0.56522, HEC SOV: 0.78333
d1a4sa_: P: 0.598, R: 0.598, F: 0.598, S: 0.46030, HEC ACC: 0.37942, HEC SOV: 0.65601
d1a6da1: P: 0.359, R: 0.371, F: 0.365, S: 0.28491, HEC ACC: 0.45714, HEC SOV: 0.51902
d1a8l_1: P: 0.931, R: 0.931, F: 0.931, S: 0.71125, HEC ACC: 0.52212, HEC SOV: 0.79980
d1af2a1: P: 0.667, R: 0.677, F: 0.672, S: 0.53294, HEC ACC: 0.37415, HEC SOV: 0.60296
d1afwb1: P: 0.809, R: 0.809, F: 0.809, S: 0.44698, HEC ACC: 0.34606, HEC SOV: 0.53551
d1ahja_: P: 0.937, R: 0.937, F: 0.937, S: 0.34364, HEC ACC: 0.15152, HEC SOV: 0.41875
d1ahua1: P: 0.613, R: 0.613, F: 0.613, S: 0.36534, HEC ACC: 0.37631, HEC SOV: 0.47661
d1ai3__: P: 0.408, R: 0.520, F: 0.457, S: 0.33924, HEC ACC: 0.35749, HEC SOV: 0.48713
d1aj8a_: P: 0.798, R: 0.784, F: 0.791, S: 0.42612, HEC ACC: 0.37197, HEC SOV: 0.44400
d1ako__: P: 0.827, R: 0.835, F: 0.831, S: 0.47936, HEC

In [8]:
malisam_tms = test('malisam')

d1a05a_d1dgsa3: P: 0.348, R: 0.312, F: 0.329, S: 0.25622, HEC ACC: 0.35227, HEC SOV: 0.50427
d1a05a_d1j71a_: P: 0.016, R: 0.018, F: 0.017, S: 0.15449, HEC ACC: 0.27053, HEC SOV: 0.42084
d1a05a_d1rblm_: P: 0.543, R: 0.557, F: 0.550, S: 0.39318, HEC ACC: 0.45026, HEC SOV: 0.72338
d1a2za_d1ghha_: P: 0.221, R: 0.224, F: 0.222, S: 0.27555, HEC ACC: 0.44318, HEC SOV: 0.62232
d1a2za_d1u9da_: P: 0.128, R: 0.150, F: 0.138, S: 0.27351, HEC ACC: 0.40000, HEC SOV: 0.55858
d1a7j__d1kafa_: P: 0.507, R: 0.679, F: 0.580, S: 0.37802, HEC ACC: 0.37838, HEC SOV: 0.54415
d1a7j__d2if1__: P: 0.579, R: 0.698, F: 0.633, S: 0.36007, HEC ACC: 0.25616, HEC SOV: 0.45418
d1aa7a_d1b68a_: P: 0.013, R: 0.013, F: 0.013, S: 0.18596, HEC ACC: 0.42791, HEC SOV: 0.61475
d1aa7a_d1qkra_: P: 0.000, R: 0.000, F: 0.000, S: 0.21590, HEC ACC: 0.35766, HEC SOV: 0.52206
d1ac5__d1jroa3: P: 0.197, R: 0.194, F: 0.196, S: 0.24291, HEC ACC: 0.27835, HEC SOV: 0.42084
d1ac5__d1vk0a_: P: 0.000, R: 0.000, F: 0.000, S: 0.13467, HEC ACC: 0.0